# Spam & Ham classifier

## 1. Imports and setup

In [15]:
import pandas as pd
import numpy

import re
import nltk
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, ConfusionMatrixDisplay
)

print("All libraries imported successfully.")

All libraries imported successfully.


In [16]:
lemmatizer = WordNetLemmatizer()

In [17]:
def get_wordnet_pos(word):
    """Map POS tag to first character lemmatize() accepts"""
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}

    return tag_dict.get(tag, wordnet.NOUN)

In [18]:
messages = pd.read_csv('../data/sms_spam_collection/sms_collection', sep='\t',names=['label', 'message'])
print(f"Dataset shape: {messages.shape[0]} rows, {messages.shape[1]} columns\n")
print(messages.head())

Dataset shape: 5572 rows, 2 columns

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [19]:
print(messages["label"].value_counts())

label
ham     4825
spam     747
Name: count, dtype: int64


In [20]:
messages["target"] = messages["label"].map({"ham": 0, "spam": 1})
messages[["label", "target"]].head()

,label,target
0,ham,0
1,ham,0
2,spam,1
3,ham,0
4,ham,0


In [21]:
# go thru each message -> remove non-alphabetic characters -> lowercase -> tokenize -> remove stopwords -> lemmatize -> join back to string
corpus = []
for i in range(len(messages)):
    message = re.sub('[^a-zA-Z]', ' ', messages['message'][i])
    message = message.lower()
    message = nltk.word_tokenize(message)
    tagged = nltk.pos_tag(message)
    message = [lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag)) for word, tag in tagged if word not in set(stopwords.words('english'))]
    corpus.append(' '.join(message))

In [22]:
corpus[:5]  # show first 5 preprocessed messages

['go jurong point crazy available bugis n great world la e buffet cine got amore wat',
 'ok lar joking wif u oni',
 'free entry wkly comp win fa cup final tkts st may text fa receive entry question std txt rate c apply',
 'u dun say early hor u c already say',
 'nah think go usf life around though']

In [24]:
y = messages["target"]

X_train, X_test, y_train, y_test = train_test_split(
    corpus, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y # keep spam/ham ratio same in both splits
    )


Stratified splitting is important because spam/ham datasets are usually imbalanced.

## 6. Pipeline: BoW + MultinomialNB

In [25]:
model = Pipeline([
    ("bow", CountVectorizer(
        lowercase=True,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2,             # ignore very rare words
        max_df=0.95           # ignore very common words
    )),
    ("nb", MultinomialNB(alpha=1.0))  # Laplace smoothing
])

model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('bow', ...), ('nb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",'english'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float in range [0.0, 1.0] or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float, the parameter represents a proportion of documents, integerabsolute counts.This parameter is ignored if vocabulary is not None.",0.95
,"min_df min_df: float in range [0.0, 1.0] or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float, the parameter represents a proportion of documents, integerabsolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If

In [26]:
y_pred = model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1-score :", f1_score(y_test, y_pred))

print(classification_report(y_test, y_pred, target_names=["ham", "spam"]))

cm = confusion_matrix(y_test, y_pred)

Accuracy : 0.979372197309417
Precision: 0.9436619718309859
Recall   : 0.8993288590604027
F1-score : 0.9209621993127147
              precision    recall  f1-score   support

         ham       0.98      0.99      0.99       966
        spam       0.94      0.90      0.92       149

    accuracy                           0.98      1115
   macro avg       0.96      0.95      0.95      1115
weighted avg       0.98      0.98      0.98      1115



In [33]:
examples = [
    "SMS SERVICES. for your inclusive text credits, pls goto www.comuk.net login= 3qxj9 unsubscribe with STOP, no extra charge. help 08702840625.COMUK. 220-CM2 9AE",
    "Why did I wake up on my own >:(",
    "Now get step 2 outta the way. Congrats again.",
    "Congratulations ur awarded either å£500 of CD gift vouchers & Free entry 2 å£100 weekly draw txt MUSIC to 87066 TnCs www.Ldew.com1win150ppmx3age16",
    "I have a date on Sunday with Will!!",
    "You have won a lottery prize of $1000! Call 123-456-7890 to claim your prize.",
    "Hey, are we still meeting for lunch today?"
]

preds = model.predict(examples)
probs = model.predict_proba(examples)

for text, pred, prob in zip(examples, preds, probs):
    label = "spam" if pred == 1 else "ham"
    print(f"{'Spam' if pred == 1 else 'Ham'} (spam prob={prob[1]:.3f}) -> {text[:60]}...")

Spam (spam prob=1.000) -> SMS SERVICES. for your inclusive text credits, pls goto www....
Ham (spam prob=0.012) -> Why did I wake up on my own >:(...
Ham (spam prob=0.002) -> Now get step 2 outta the way. Congrats again....
Spam (spam prob=1.000) -> Congratulations ur awarded either å£500 of CD gift vouchers ...
Ham (spam prob=0.042) -> I have a date on Sunday with Will!!...
Spam (spam prob=1.000) -> You have won a lottery prize of $1000! Call 123-456-7890 to ...
Ham (spam prob=0.000) -> Hey, are we still meeting for lunch today?...
